In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

import numpy as np
import pandas as pd

from sklearn.decomposition import KernelPCA
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Restart kernel trước khi chạy cell này nếu trước đó đã lỗi nhiều lần
spark = (
    SparkSession.builder
    .appName("TaxiPipeline")
    .master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.default.parallelism", "32")
    .getOrCreate()
)

df = spark.read.parquet('/user/data/raw/*.parquet')

print(f"Number of raw parquet files: {len(df.inputFiles())}")
print(f"Total number of rows: {df.count()}")
df.printSchema()

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a0f597d7-b0a3-4f5a-abc0-a3f38d6107a8;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 200ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	-----------------------------------------------

Number of raw parquet files: 12


Total number of rows: 48722602
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [2]:
df_input = df.filter(
    (F.col("tpep_pickup_datetime") >= F.lit("2025-01-01 00:00:00")) &
    (F.col("tpep_pickup_datetime") < F.lit("2026-01-01 00:00:00"))
)
print("df_input rows:", df_input.count())

df_input.select(
    F.min("tpep_pickup_datetime").alias("min_time"),
    F.max("tpep_pickup_datetime").alias("max_time")
).show(truncate=False)


df_input rows: 48722573


+-------------------+-------------------+
|min_time           |max_time           |
+-------------------+-------------------+
|2025-01-01 00:00:00|2025-12-31 23:59:59|
+-------------------+-------------------+



In [3]:
SLOT_MINUTES = 30
MIN_MEAN_PICKUPS_PER_SLOT = 10.0
N_CLUSTERS = 8
CLUSTER_LAG = 12
DISAGG_LAG = 12
TRAIN_RATIO = 0.8
RANDOM_STATE = 42

RF_CLUSTER_PARAMS = dict(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=2,
    n_jobs=1,
    random_state=RANDOM_STATE
)

RF_DISAGG_PARAMS = dict(
    n_estimators=50,
    max_depth=10,
    min_samples_leaf=2,
    n_jobs=1,
    random_state=RANDOM_STATE
)

In [4]:
def preprocess_pickup_counts(
    df,
    slot_minutes: int = 30,
    pickup_col: str = "tpep_pickup_datetime",
    zone_col: str = "PULocationID"
):
    """
    Dùng đúng cho schema hiện tại:
    - tpep_pickup_datetime đang là timestamp, không chia /1000
    - chỉ giữ cột cần thiết để nhẹ RAM hơn
    """
    df_small = df.select(pickup_col, zone_col)

    sdf = (
        df_small
        .filter(F.col(pickup_col).isNotNull())
        .filter(F.col(zone_col).isNotNull())
        .withColumn(zone_col, F.col(zone_col).cast("int"))
        .filter(F.col(zone_col) > 0)
        .withColumn("pickup_ts", F.col(pickup_col).cast("timestamp"))
        .filter(F.col("pickup_ts").isNotNull())
    )

    slot_seconds = slot_minutes * 60

    sdf = (
        sdf
        .withColumn(
            "slot_unix",
            (F.floor(F.unix_timestamp("pickup_ts") / slot_seconds) * slot_seconds).cast("long")
        )
        .withColumn("slot_ts", F.to_timestamp(F.from_unixtime(F.col("slot_unix"))))
    )

    pickup_counts = (
        sdf.groupBy("slot_ts", zone_col)
           .agg(F.count(F.lit(1)).alias("pickup_count"))
           .withColumnRenamed(zone_col, "zone_id")
           .select("slot_ts", "zone_id", "pickup_count")
    )

    return pickup_counts


def build_dense_zone_slot_matrix_light(pickup_counts_sdf, slot_minutes=30):
    """
    Bản nhẹ RAM:
    - Spark chỉ aggregate
    - kéo aggregate về pandas
    - pivot trong pandas
    - reindex full timeline trong pandas
    """
    pdf = (
        pickup_counts_sdf
        .orderBy("slot_ts", "zone_id")
        .toPandas()
    )

    if pdf.empty:
        raise ValueError("pickup_counts_sdf is empty")

    pdf["slot_ts"] = pd.to_datetime(pdf["slot_ts"])
    pdf["zone_id"] = pdf["zone_id"].astype(int)
    pdf["pickup_count"] = pdf["pickup_count"].astype(np.float32)

    pivot_pdf = pdf.pivot_table(
        index="slot_ts",
        columns="zone_id",
        values="pickup_count",
        aggfunc="sum",
        fill_value=0
    )

    full_index = pd.date_range(
        start=pivot_pdf.index.min(),
        end=pivot_pdf.index.max(),
        freq=f"{slot_minutes}min"
    )

    pivot_pdf = pivot_pdf.reindex(full_index, fill_value=0)
    pivot_pdf.index.name = "slot_ts"
    pivot_pdf = pivot_pdf.sort_index(axis=1)

    return pivot_pdf.astype(np.float32)


def filter_active_zones(pivot_pdf: pd.DataFrame, min_mean_pickups_per_slot: float = 10.0):
    zone_means = pivot_pdf.mean(axis=0)
    active_zones = zone_means[zone_means >= min_mean_pickups_per_slot].index.tolist()
    filtered = pivot_pdf[active_zones].copy()
    return filtered, zone_means.sort_values(ascending=False)


def compute_correlation_dissimilarity(zone_ts_pdf: pd.DataFrame):
    corr_df = zone_ts_pdf.corr(method="pearson")
    corr_df = corr_df.fillna(0.0).clip(-1.0, 1.0)
    np.fill_diagonal(corr_df.values, 1.0)
    dist_df = 1.0 - corr_df
    return corr_df, dist_df


def gaussian_kernel_on_distance_rows(dist_df: pd.DataFrame):
    D = dist_df.values.astype(np.float64)
    L = D.shape[0]

    row_sq = np.sum(D * D, axis=1, keepdims=True)
    sq_euclidean = row_sq + row_sq.T - 2 * (D @ D.T)
    sq_euclidean = np.maximum(sq_euclidean, 0.0)

    G = np.exp(-sq_euclidean / max(L, 1))
    G_df = pd.DataFrame(G, index=dist_df.index, columns=dist_df.columns)
    return G_df


def cluster_zones_kernel_pca_kmeans(
    dist_df: pd.DataFrame,
    n_clusters: int = 8,
    random_state: int = 42
):
    G_df = gaussian_kernel_on_distance_rows(dist_df)

    n_components = min(max(n_clusters, 2), max(1, G_df.shape[0] - 1))
    kpca = KernelPCA(
        n_components=n_components,
        kernel="precomputed",
        random_state=random_state
    )
    embedding = kpca.fit_transform(G_df.values)

    kmeans = KMeans(
        n_clusters=n_clusters,
        random_state=random_state,
        n_init=20
    )
    labels = kmeans.fit_predict(embedding)

    cluster_map = pd.Series(labels, index=dist_df.index, name="cluster_id")
    return cluster_map, G_df, embedding


def aggregate_cluster_demand(zone_ts_pdf: pd.DataFrame, cluster_map: pd.Series):
    cluster_ts = {}
    for c in sorted(cluster_map.unique()):
        zones = cluster_map[cluster_map == c].index.tolist()
        cluster_ts[c] = zone_ts_pdf[zones].sum(axis=1)

    cluster_ts_pdf = pd.DataFrame(cluster_ts, index=zone_ts_pdf.index).sort_index(axis=1)
    cluster_ts_pdf.columns = [f"cluster_{c}" for c in cluster_ts_pdf.columns]
    return cluster_ts_pdf


def make_lagged_supervised(series: np.ndarray, lag: int):
    X, y = [], []
    for t in range(lag, len(series)):
        X.append(series[t-lag:t][::-1])  # newest -> oldest
        y.append(series[t])
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32)


def chronological_train_test_split(X, y, train_ratio=0.8):
    n = len(X)
    split = int(n * train_ratio)
    return X[:split], X[split:], y[:split], y[split:]


def safe_mape(y_true, y_pred, eps=1e-6):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    denom = np.maximum(np.abs(y_true), eps)
    return np.mean(np.abs(y_true - y_pred) / denom) * 100.0


def regression_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "MAPE": float(safe_mape(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred))
    }


def fit_predict_cluster_models(
    cluster_ts_pdf: pd.DataFrame,
    lag: int = 12,
    train_ratio: float = 0.8,
    rf_params: dict = None
):
    if rf_params is None:
        rf_params = RF_CLUSTER_PARAMS

    cluster_models = {}
    cluster_predictions = {}
    cluster_metrics = {}
    cluster_test_indices = {}

    for col in cluster_ts_pdf.columns:
        s = cluster_ts_pdf[col].values.astype(np.float32)

        X, y = make_lagged_supervised(s, lag=lag)
        X_train, X_test, y_train, y_test = chronological_train_test_split(X, y, train_ratio)

        x_scaler = MinMaxScaler()
        y_scaler = MinMaxScaler()

        X_train_sc = x_scaler.fit_transform(X_train)
        X_test_sc = x_scaler.transform(X_test)

        y_train_sc = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()

        model = RandomForestRegressor(**rf_params)
        model.fit(X_train_sc, y_train_sc)

        y_pred_sc = model.predict(X_test_sc)
        y_pred = y_scaler.inverse_transform(y_pred_sc.reshape(-1, 1)).ravel()

        y_pred = np.round(np.clip(y_pred, 0, None))

        metrics = regression_metrics(y_test, y_pred)

        cluster_models[col] = {
            "model": model,
            "x_scaler": x_scaler,
            "y_scaler": y_scaler
        }
        cluster_predictions[col] = {
            "y_test": y_test,
            "y_pred": y_pred
        }
        cluster_metrics[col] = metrics

        all_target_times = cluster_ts_pdf.index[lag:]
        split_idx = int(len(all_target_times) * train_ratio)
        cluster_test_indices[col] = all_target_times[split_idx:]

    return cluster_models, cluster_predictions, cluster_metrics, cluster_test_indices


def build_disaggregation_train_dataset(
    cluster_series: pd.Series,
    zone_ts_pdf: pd.DataFrame,
    zones_in_cluster,
    disagg_lag: int = 12,
    train_ratio: float = 0.8
):
    """
    Train disaggregation bằng actual cluster demand history.
    """
    cluster_values = cluster_series.values.astype(np.float32)
    zone_values = zone_ts_pdf[zones_in_cluster].values.astype(np.float32)
    time_index = cluster_series.index

    X_all, Y_all, T_all = [], [], []
    for t in range(disagg_lag - 1, len(cluster_values)):
        x = cluster_values[t - disagg_lag + 1:t + 1][::-1]
        y = zone_values[t]
        X_all.append(x)
        Y_all.append(y)
        T_all.append(time_index[t])

    X_all = np.asarray(X_all, dtype=np.float32)
    Y_all = np.asarray(Y_all, dtype=np.float32)
    T_all = np.asarray(T_all)

    split_idx = int(len(X_all) * train_ratio)

    X_train = X_all[:split_idx]
    Y_train = Y_all[:split_idx]
    X_test_dummy = X_all[split_idx:]
    Y_test_dummy = Y_all[split_idx:]
    T_test_dummy = T_all[split_idx:]

    return X_train, X_test_dummy, Y_train, Y_test_dummy, T_test_dummy


def replace_test_inputs_with_predicted_cluster_history(
    full_cluster_actual: pd.Series,
    cluster_test_pred: np.ndarray,
    disagg_lag: int,
    train_ratio: float,
    cluster_lag: int
):
    """
    Align predicted cluster demand vào đúng timeline gốc.

    cluster_test_pred sinh ra từ supervised dataset với cluster_lag,
    nên prediction test bắt đầu ở original index:
        pred_start_idx = cluster_lag + split_idx_targets
    """
    actual = full_cluster_actual.values.astype(np.float32)
    time_index = full_cluster_actual.index
    n_total = len(actual)

    n_targets = n_total - cluster_lag
    split_idx_targets = int(n_targets * train_ratio)
    pred_start_idx = cluster_lag + split_idx_targets

    stitched = actual.copy()

    n_replace = min(len(cluster_test_pred), n_total - pred_start_idx)
    stitched[pred_start_idx:pred_start_idx + n_replace] = cluster_test_pred[:n_replace]

    X_test = []
    T_test = []

    test_start_idx = max(pred_start_idx, disagg_lag - 1)
    test_end_idx = pred_start_idx + n_replace

    for t in range(test_start_idx, test_end_idx):
        x = stitched[t - disagg_lag + 1:t + 1][::-1]
        X_test.append(x)
        T_test.append(time_index[t])

    return np.asarray(X_test, dtype=np.float32), np.asarray(T_test)


def fit_disaggregation_models(
    zone_ts_pdf: pd.DataFrame,
    cluster_ts_pdf: pd.DataFrame,
    cluster_map: pd.Series,
    cluster_predictions: dict,
    disagg_lag: int = 12,
    cluster_lag: int = 12,
    train_ratio: float = 0.8,
    rf_params: dict = None
):
    if rf_params is None:
        rf_params = RF_DISAGG_PARAMS

    disagg_models = {}
    zone_level_predictions = {}
    zone_level_metrics = {}

    unique_clusters = sorted(cluster_map.unique())

    for c in unique_clusters:
        cluster_col = f"cluster_{c}"
        zones = cluster_map[cluster_map == c].index.tolist()
        cluster_series = cluster_ts_pdf[cluster_col]

        X_train, _, Y_train, _, _ = build_disaggregation_train_dataset(
            cluster_series=cluster_series,
            zone_ts_pdf=zone_ts_pdf,
            zones_in_cluster=zones,
            disagg_lag=disagg_lag,
            train_ratio=train_ratio
        )

        y_pred_cluster = cluster_predictions[cluster_col]["y_pred"]

        X_test, T_test_pred = replace_test_inputs_with_predicted_cluster_history(
            full_cluster_actual=cluster_series,
            cluster_test_pred=np.asarray(y_pred_cluster, dtype=np.float32),
            disagg_lag=disagg_lag,
            train_ratio=train_ratio,
            cluster_lag=cluster_lag
        )

        test_zone_df = zone_ts_pdf.loc[T_test_pred, zones]
        Y_test = test_zone_df.values.astype(np.float32)

        x_scaler = MinMaxScaler()
        y_scaler = MinMaxScaler()

        X_train_sc = x_scaler.fit_transform(X_train)
        X_test_sc = x_scaler.transform(X_test)
        Y_train_sc = y_scaler.fit_transform(Y_train)

        base_rf = RandomForestRegressor(**rf_params)
        # model = MultiOutputRegressor(base_rf, n_jobs=-1)
        model = MultiOutputRegressor(base_rf, n_jobs=1)
        model.fit(X_train_sc, Y_train_sc)

        Y_pred_sc = model.predict(X_test_sc)
        Y_pred = y_scaler.inverse_transform(Y_pred_sc)
        Y_pred = np.round(np.clip(Y_pred, 0, None))

        per_zone_metrics = {}
        for i, z in enumerate(zones):
            per_zone_metrics[z] = regression_metrics(Y_test[:, i], Y_pred[:, i])

        avg_metrics = {
            "MAE": float(np.mean([m["MAE"] for m in per_zone_metrics.values()])),
            "RMSE": float(np.mean([m["RMSE"] for m in per_zone_metrics.values()])),
            "MAPE": float(np.mean([m["MAPE"] for m in per_zone_metrics.values()])),
            "R2": float(np.mean([m["R2"] for m in per_zone_metrics.values()])),
        }

        disagg_models[c] = {
            "model": model,
            "x_scaler": x_scaler,
            "y_scaler": y_scaler,
            "zones": zones
        }

        zone_level_predictions[c] = {
            "times": T_test_pred,
            "zones": zones,
            "y_test": Y_test,
            "y_pred": Y_pred
        }

        zone_level_metrics[c] = {
            "avg_metrics": avg_metrics,
            "per_zone_metrics": per_zone_metrics
        }

    return disagg_models, zone_level_predictions, zone_level_metrics



In [5]:
required_names = [
    "df_input",
    "preprocess_pickup_counts",
    "build_dense_zone_slot_matrix_light",
    "replace_test_inputs_with_predicted_cluster_history",
    "fit_disaggregation_models",
]

for name in required_names:
    print(name, "OK" if name in globals() else "MISSING")

df_input OK
preprocess_pickup_counts OK
build_dense_zone_slot_matrix_light OK
replace_test_inputs_with_predicted_cluster_history OK
fit_disaggregation_models OK


In [6]:
print("Step 1/8: preprocess pickup counts...")
pickup_counts_sdf = preprocess_pickup_counts(df_input).cache()
pickup_counts_sdf.count()

print("Step 2/8: build dense zone-slot matrix...")
pivot_pdf = build_dense_zone_slot_matrix_light(pickup_counts_sdf)
print("Dense matrix shape:", pivot_pdf.shape)

print("Step 3/8: filter active zones...")
zone_ts_pdf, zone_means = filter_active_zones(
    pivot_pdf,
    min_mean_pickups_per_slot=MIN_MEAN_PICKUPS_PER_SLOT
)
print("Filtered matrix shape:", zone_ts_pdf.shape)
print("Number of active zones:", zone_ts_pdf.shape[1])

print("Step 4/8: compute correlation and dissimilarity...")
corr_df, dist_df = compute_correlation_dissimilarity(zone_ts_pdf)

print("Step 5/8: cluster zones...")
cluster_map, kernel_df, embedding = cluster_zones_kernel_pca_kmeans(
    dist_df=dist_df,
    n_clusters=N_CLUSTERS,
    random_state=RANDOM_STATE
)
print(cluster_map.value_counts().sort_index())

print("Step 6/8: aggregate cluster demand...")
cluster_ts_pdf = aggregate_cluster_demand(zone_ts_pdf, cluster_map)
print("Cluster demand shape:", cluster_ts_pdf.shape)
# Giải phóng Spark trước khi chạy sklearn
try:
    pickup_counts_sdf.unpersist()
except:
    pass

try:
    spark.catalog.clearCache()
except:
    pass

try:
    spark.stop()
except:
    pass

print("Step 7/8: cluster-level forecasting...")
cluster_models, cluster_predictions, cluster_metrics, cluster_test_indices = fit_predict_cluster_models(
    cluster_ts_pdf=cluster_ts_pdf,
    lag=CLUSTER_LAG,
    train_ratio=TRAIN_RATIO,
    rf_params=RF_CLUSTER_PARAMS
)

print("\nCluster-level metrics:")
for k, v in cluster_metrics.items():
    print(k, v)

print("\nStep 8/8: disaggregation to zone-level...")
disagg_models, zone_level_predictions, zone_level_metrics = fit_disaggregation_models(
    zone_ts_pdf=zone_ts_pdf,
    cluster_ts_pdf=cluster_ts_pdf,
    cluster_map=cluster_map,
    cluster_predictions=cluster_predictions,
    disagg_lag=DISAGG_LAG,
    cluster_lag=CLUSTER_LAG,
    train_ratio=TRAIN_RATIO,
    rf_params=RF_DISAGG_PARAMS
)

print("\nZone-level average metrics by cluster:")
for c, res in zone_level_metrics.items():
    print(f"cluster_{c}:", res["avg_metrics"])

overall_zone_metrics = {
    "MAE": float(np.mean([res["avg_metrics"]["MAE"] for res in zone_level_metrics.values()])),
    "RMSE": float(np.mean([res["avg_metrics"]["RMSE"] for res in zone_level_metrics.values()])),
    "MAPE": float(np.mean([res["avg_metrics"]["MAPE"] for res in zone_level_metrics.values()])),
    "R2": float(np.mean([res["avg_metrics"]["R2"] for res in zone_level_metrics.values()])),
}

print("\nOverall zone-level average metrics:")
print(overall_zone_metrics)

Step 1/8: preprocess pickup counts...


Step 2/8: build dense zone-slot matrix...


Dense matrix shape: (17520, 262)
Step 3/8: filter active zones...
Filtered matrix shape: (17520, 49)
Number of active zones: 49
Step 4/8: compute correlation and dissimilarity...
Step 5/8: cluster zones...
cluster_id
0    10
1    11
2     4
3     4
4     8
5     2
6     2
7     8
Name: count, dtype: int64
Step 6/8: aggregate cluster demand...
Cluster demand shape: (17520, 8)
Step 7/8: cluster-level forecasting...

Cluster-level metrics:
cluster_0 {'MAE': 42.40805254140491, 'RMSE': 61.194382317206745, 'MAPE': 10.648170028406419, 'R2': 0.9768231375516284}
cluster_1 {'MAE': 40.2395773843518, 'RMSE': 58.63386427634764, 'MAPE': 10.503153613547056, 'R2': 0.9694111935314981}
cluster_2 {'MAE': 21.623072529982867, 'RMSE': 38.64386590855853, 'MAPE': 15.189248667845698, 'R2': 0.9649486199535338}
cluster_3 {'MAE': 21.026841804683038, 'RMSE': 29.071938579886446, 'MAPE': 21.093767299662247, 'R2': 0.8544476730487301}
cluster_4 {'MAE': 30.284694460308394, 'RMSE': 44.358602415133326, 'MAPE': 12.5384248